In [3]:
import Bio
print(Bio.__version__)


1.87


In [4]:
# Change BIO to Bio here
from Bio.PDB import PDBParser, PDBIO, Select

class ProteinCleanSelect(Select):
    """Filter rule to extract ONLY the clean protein chain."""
    def accept_model(self, model): return 1
    
    def accept_chain(self, chain):
        # 6O0K contains its active Bcl-2 sequence on Chain A
        return chain.id == 'A'
        
    def accept_residue(self, residue):
        # Reject water molecules (HOH) and the native drug ligand (unwanted heterogens)
        if residue.id[0].startswith('H_') or residue.id[0] == 'W':
            return 0
        return 1

# Initialize the PDB structural reader
parser = PDBParser(QUIET=True)
structure = parser.get_structure('6O0K', '6O0K.pdb')

# Save out a clean structural file
io = PDBIO()
io.set_structure(structure)
io.save('protein_clean.pdb', select=ProteinCleanSelect())

print("=== Step 3 Complete: Clean Protein Saved as 'protein_clean.pdb' ===")
print("[✔] Removed all crystallographic waters.")
print("[✔] Stripped away native Venetoclax ligand.")
print("[✔] Isolated Chain A (Active Human Bcl-2 structure).")


=== Step 3 Complete: Clean Protein Saved as 'protein_clean.pdb' ===
[✔] Removed all crystallographic waters.
[✔] Stripped away native Venetoclax ligand.
[✔] Isolated Chain A (Active Human Bcl-2 structure).


In [5]:
from openbabel import openbabel

def convert_pdb_to_pdbqt(input_pdb, output_pdbqt):
    obConversion = openbabel.OBConversion()
    obConversion.SetInAndOutFormats("pdb", "pdbqt")
    
    mol = openbabel.OBMol()
    if obConversion.ReadFile(mol, input_pdb):
        # Automatically add polar hydrogens and compute structural charges
        mol.AddHydrogens(False, True) # (polaronly=True)
        obConversion.WriteFile(mol, output_pdbqt)
        print(f"[✔] Successfully converted protein to: '{output_pdbqt}'")
    else:
        print("[X] Failed to read protein PDB file.")

convert_pdb_to_pdbqt('protein_clean.pdb', 'protein_receptor.pdbqt')

[✔] Successfully converted protein to: 'protein_receptor.pdbqt'


In [6]:
config_content = """# AutoDock Vina Configuration File for Bcl-2 (PDB: 6O0K)

# 1. Structural Inputs
receptor = protein_receptor.pdbqt

# 2. Search Space Dimensions (Centered precisely on Venetoclax pocket)
center_x = -17.4
center_y = 7.3
center_z = -18.2

# 3. Grid Box Size (Angstroms - large enough to let your ligands rotate freely)
size_x = 22.0
size_y = 22.0
size_z = 22.0

# 4. Simulation Parameters
exhaustiveness = 8     # Default thoroughness of the global search space
num_modes = 9          # Max number of binding poses to save
energy_range = 3       # Max energy difference (kcal/mol) between top and bottom poses
"""

with open('config.txt', 'w') as f:
    f.write(config_content)

print("[✔] Successfully created 'config.txt' with precision 6O0K pocket coordinates!")

[✔] Successfully created 'config.txt' with precision 6O0K pocket coordinates!


In [7]:
import os
from vina import Vina

# Initialize Vina in Python (sf_name='vina' sets the standard scoring function)
v = Vina(sf_name='vina')

# Load the target receptor protein
v.set_receptor('protein_receptor.pdbqt')

top_cids = [156294005, 51003064, 44570051, 168291284, 156294009]

print("=== Starting Step 6: Running AutoDock Vina Simulations ===\n")

for cid in top_cids:
    ligand_file = f'candidate_{cid}.pdbqt'
    output_pose_file = f'candidate_{cid}_docked.pdbqt'
    
    if not os.path.exists(ligand_file):
        print(f"[!] Target ligand {ligand_file} missing. Skipping...")
        continue
        
    print(f"[*] Simulating docking for Candidate CID: {cid}...")
    
    # Set the current ligand to test
    v.set_ligand_from_file(ligand_file)
    
    # Instruct Vina where to look based on our config coordinates
    v.compute_vina_maps(center=[-17.4, 7.3, -18.2], box_size=[22.0, 22.0, 22.0])
    
    # Run the structural optimization search
    v.dock(exhaustiveness=8, n_poses=9)
    
    # Extract the top binding affinity score (kcal/mol)
    # score[0][0] corresponds to the tightest, lowest-energy structural binding pose
    energies = v.energies()
    top_score = energies[0][0]
    
    print(f"    [✔] Calculation finished!")
    print(f"    -> Top Binding Affinity: {top_score:.2f} kcal/mol")
    
    # Save the physical 3D spatial orientations of the docked poses
    v.write_poses(output_pose_file, n_poses=5, overwrite=True)
    print(f"    -> 3D binding coordinates saved to '{output_pose_file}'\n")

print("=== Simulation Pipeline Complete! Check your scores above! ===")

ModuleNotFoundError: No module named 'vina'

In [8]:
input_filename = "protein_receptor.pdbqt"
output_filename = "protein_receptor_fixed.pdbqt"

print(f"[*] Beginning deep scan of {input_filename}...")

clean_lines = []
inside_root_block = False
blocks_removed = 0

with open(input_filename, "r") as infile:
    for line in infile:
        # Check if a problematic block is starting
        if line.strip() == "ROOT":
            inside_root_block = True
            blocks_removed += 1
            continue
            
        # Check if the problematic block is ending
        if line.strip() == "ENDROOT":
            inside_root_block = False
            continue
            
        # Only keep the line if it is outside of any ROOT/ENDROOT blocks
        if not inside_root_block:
            clean_lines.append(line)

# Save the completely cleared records to the new file
with open(output_filename, "w") as outfile:
    outfile.writelines(clean_lines)

print(f"[✔] Successfully sanitized receptor configuration!")
print(f"    -> Identified and destroyed {blocks_removed} illegal ligand ROOT blocks.")
print(f"    -> Saved ready-to-dock structure as: '{output_filename}'")

[*] Beginning deep scan of protein_receptor.pdbqt...
[✔] Successfully sanitized receptor configuration!
    -> Identified and destroyed 105 illegal ligand ROOT blocks.
    -> Saved ready-to-dock structure as: 'protein_receptor_fixed.pdbqt'


In [9]:
input_filename = "protein_receptor.pdbqt"
output_filename = "protein_receptor_fixed.pdbqt"

print(f"[*] Starting complete block-level sweep of {input_filename}...")

clean_lines = []
skip_mode = False
blocks_removed = 0
lines_skipped = 0

with open(input_filename, "r") as infile:
    for line in infile:
        cleaned_line = line.strip()
        
        # Detect the start of an invalid ligand block
        if cleaned_line == "ROOT":
            skip_mode = True
            blocks_removed += 1
            lines_skipped += 1
            continue
            
        # Detect the end of an invalid ligand block
        if cleaned_line == "ENDROOT":
            skip_mode = False
            lines_skipped += 1
            continue
            
        # If we are inside a ROOT block, skip EVERYTHING (including BRANCH tags)
        if skip_mode:
            lines_skipped += 1
            continue
            
        # Keep everything else
        clean_lines.append(line)

# Save the pristine receptor file
with open(output_filename, "w") as outfile:
    outfile.writelines(clean_lines)

print(f"[✔] Deep clean complete!")
print(f"    -> Isolated and dropped {blocks_removed} complete torsion blocks.")
print(f"    -> Purged a total of {lines_skipped} junk formatting lines.")
print(f"    -> Your true rigid receptor is ready at: '{output_filename}'")

[*] Starting complete block-level sweep of protein_receptor.pdbqt...
[✔] Deep clean complete!
    -> Isolated and dropped 105 complete torsion blocks.
    -> Purged a total of 490 junk formatting lines.
    -> Your true rigid receptor is ready at: 'protein_receptor_fixed.pdbqt'


In [12]:
from rdkit import Chem

# Read your optimized SDF molecule
sdf_file = "candidate_156294005_optimized.sdf"
mol = next(Chem.SDMolSupplier(sdf_file, removeHs=False))

if mol:
    # Save it cleanly as a .mol2 file
    Chem.MolToMol2File(mol, "candidate_156294005.mol2")
    print("[✔] Successfully converted ligand to 'candidate_156294005.mol2'")

AttributeError: module 'rdkit.Chem' has no attribute 'MolToMol2File'

In [14]:
!pip install vina

  Using cached vina-1.2.7.tar.gz (89 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [77 lines of output]
      fatal: not a git repository (or any of the parent directories): .git
      Version found 1.2.7 (from __init__.py)
      C:\Users\Adnan Habib\AppData\Local\Temp\pip-build-env-zc3rm00y\overlay\Lib\site-packages\setuptools\dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license